In [7]:
import os

WORDLIST_PATH = r"C:\Users\HP\Desktop\code\rockyou-10k.txt"

if not os.path.exists(WORDLIST_PATH):
    raise FileNotFoundError(
        f"Le fichier {WORDLIST_PATH!r} est introuvable. "
        f"Téléchargez-le depuis le lien donné dans la cellule précédente "
        f"et placez-le dans le même dossier que ce notebook."
    )

with open(WORDLIST_PATH, "r", encoding="utf-8", errors="ignore") as f:
    words = [w.strip() for w in f if w.strip()]

print(f"Wordlist : {WORDLIST_PATH} ({os.path.getsize(WORDLIST_PATH)} octets, {len(words)} mots)")
print("Aperçu :", words[:10])



Wordlist : C:\Users\HP\Desktop\code\rockyou-10k.txt (73017 octets, 10000 mots)
Aperçu : ['password', '123456', '12345678', '1234', 'qwerty', '12345', 'dragon', 'pussy', 'baseball', 'football']


In [8]:
import hashlib
import time

def md5_hex(password: str) -> str:
    return hashlib.md5(password.encode("utf-8")).hexdigest()

def sha256_hex(password: str) -> str:
    return hashlib.sha256(password.encode("utf-8")).hexdigest()

# Base "volée" : 4 utilisateurs
victims = {
    "alice":   md5_hex("password"),
    "bob":     md5_hex("123456"),
    "charlie": sha256_hex("iloveyou"),
    "diana":   sha256_hex("Tr0ub4dor&3"),  # pas dans le top 10k
}

for user, h in victims.items():
    print(f"{user:8s} -> {h}")

alice    -> 5f4dcc3b5aa765d61d8327deb882cf99
bob      -> e10adc3949ba59abbe56e057f20f883e
charlie  -> e4ad93ca07acb8d908a3aa41e920ea4f4ef4f26e7f86cf8291c5db289780a5ae
diana    -> 48486e1514e842346ff405b1e45f44059ae82619f2306f99d0940dcb386e91f7


In [9]:
from typing import List, Tuple, Optional

def crack(target_hash: str, algo: str, wordlist: List[str]) -> Tuple[Optional[str], float, int]:
    """Retourne (mot_de_passe | None, durée_secondes, nb_essais)."""
    hash_fn = md5_hex if algo == "md5" else sha256_hex
    start = time.perf_counter()

    for i, candidate in enumerate(wordlist, 1):
        if hash_fn(candidate) == target_hash:
            return candidate, time.perf_counter() - start, i

    return None, time.perf_counter() - start, len(wordlist)


targets = [
    ("alice",   "md5",    victims["alice"]),
    ("bob",     "md5",    victims["bob"]),
    ("charlie", "sha256", victims["charlie"]),
    ("diana",   "sha256", victims["diana"]),
]

print(f"{'user':10s} {'algo':8s} {'résultat':20s} {'essais':>8s} {'temps (s)':>12s}")
print("-" * 64)

for user, algo, h in targets:
    pwd, dt, tries = crack(h, algo, words)
    print(f"{user:10s} {algo:8s} {str(pwd):20s} {tries:>8d} {dt:>12.4f}")

user       algo     résultat               essais    temps (s)
----------------------------------------------------------------
alice      md5      password                    1       0.0000
bob        md5      123456                      2       0.0000
charlie    sha256   iloveyou                  105       0.0002
diana      sha256   None                    10000       0.0090


In [10]:
N = 200_000

def benchmark(hash_fn, n=N):
    start = time.perf_counter()
    for i in range(n):
        hash_fn(f"test{i}")
    dt = time.perf_counter() - start
    return n / dt, dt

for name, fn in [("MD5", md5_hex), ("SHA-256", sha256_hex)]:
    rate, dt = benchmark(fn)
    print(f"{name:8s} : {rate:>12,.0f} hashs/s   ({N:,} hashs en {dt:.3f} s)")

MD5      :      696,253 hashs/s   (200,000 hashs en 0.287 s)
SHA-256  :      710,474 hashs/s   (200,000 hashs en 0.282 s)


In [11]:
import os

def salted_sha256(password: str, salt: bytes) -> str:
    return hashlib.sha256(salt + password.encode("utf-8")).hexdigest()

def gen_salt(n: int = 16) -> bytes:
    return os.urandom(n)

# On simule une base de N utilisateurs ayant tous un mot de passe "faible"
import random
random.seed(42)
N_USERS = 5
salted_db = []
for i in range(N_USERS):
    pwd = random.choice(words[:100])   # un mot fréquent
    salt = gen_salt()
    salted_db.append({
        "user": f"user{i}",
        "salt": salt,
        "hash": salted_sha256(pwd, salt),
        "_clear": pwd,                  # pour vérification (ne stockez JAMAIS ça en vrai)
    })

for row in salted_db:
    print(f"{row['user']}  salt={row['salt'].hex()[:16]}...  hash={row['hash'][:16]}...  (clair: {row['_clear']})")

user0  salt=b89e59035f906495...  hash=5c6c06521d55727b...  (clair: austin)
user1  salt=633242942e0b24c7...  hash=e02cb14f86d2fcba...  (clair: mustang)
user2  salt=96bc085c31b77726...  hash=d92dfce8f3a9f306...  (clair: 1234)
user3  salt=75c635411136e708...  hash=3f76bf36606b2f39...  (clair: yellow)
user4  salt=01bdf6edb7d454e1...  hash=9209d483b8dac41b...  (clair: fuck)


In [12]:
def crack_salted(target_hash: str, salt: bytes, wordlist: list[str]) -> tuple[str | None, float]:
    start = time.perf_counter()
    for candidate in wordlist:
        if salted_sha256(candidate, salt) == target_hash:
            return candidate, time.perf_counter() - start
    return None, time.perf_counter() - start

# Attaque "ciblée" sur 1 compte
t0 = time.perf_counter()
pwd, dt1 = crack_salted(salted_db[0]["hash"], salted_db[0]["salt"], words)
print(f"1 compte salé : {pwd!r} cassé en {dt1:.3f}s")

# Attaque sur les N comptes : il faut refaire le wordlist pour chaque sel
t0 = time.perf_counter()
results = []
for row in salted_db:
    pwd, dt = crack_salted(row["hash"], row["salt"], words)
    results.append((row["user"], pwd, dt))
total = time.perf_counter() - t0

print(f"\n{N_USERS} comptes salés : total {total:.3f}s (≈ {total/N_USERS:.3f}s/compte)")
for user, pwd, dt in results:
    print(f"  {user} -> {pwd!r}  ({dt:.3f}s)")

print(f"\nFacteur multiplicatif vs 1 passe : ~{N_USERS}x")
print("Sans sel, une seule passe de wordlist aurait cassé les N hashs identiques en une fois.")

1 compte salé : 'austin' cassé en 0.000s

5 comptes salés : total 0.001s (≈ 0.000s/compte)
  user0 -> 'austin'  (0.000s)
  user1 -> 'mustang'  (0.000s)
  user2 -> '1234'  (0.000s)
  user3 -> 'yellow'  (0.000s)
  user4 -> 'fuck'  (0.000s)

Facteur multiplicatif vs 1 passe : ~5x
Sans sel, une seule passe de wordlist aurait cassé les N hashs identiques en une fois.


In [15]:
# Si nécessaire : !pip install bcrypt
import bcrypt

password = b"password"

# Génération (le sel est dans la sortie)
for rounds in (4, 8, 10, 12):
    start = time.perf_counter()
    h = bcrypt.hashpw(password, bcrypt.gensalt(rounds=rounds))
    dt = time.perf_counter() - start
    print(f"rounds={rounds:2d}  temps={dt*1000:7.1f} ms  hash={h.decode()}")

rounds= 4  temps=    1.4 ms  hash=$2b$04$Pd2tykPEAALcjmJ0eu27pOpaF9hWvQZhjIa6.xUddogU/QGdyEVQa
rounds= 8  temps=   15.7 ms  hash=$2b$08$Lyq4aeHvXcqeGykPOIGQKO8aTMSYTZKrs7Aj2R9RuTjqgzoxGtkdC
rounds=10  temps=   73.0 ms  hash=$2b$10$7rc.Pcde9ghafHu7PBx8fuTz9Q.uZ5wAwDFBoJYQetNgT.fOhXWRK
rounds=12  temps=  310.7 ms  hash=$2b$12$2Rnqb.bIPQUheaLsUn3FFOILzYD/e5i9Zt.JbQj14hv1UL0yER25m


In [19]:
# Tentative de cassage bcrypt : on choisit rounds=8 (faible) et un sous-ensemble du wordlist
ROUNDS = 8
target_clear = "iloveyou"
bcrypt_hash = bcrypt.hashpw(target_clear.encode(), bcrypt.gensalt(rounds=ROUNDS))
print(f"Cible : {bcrypt_hash.decode()}")

SUBSET = 500   # 500 mots seulement, sinon c'est trop long
start = time.perf_counter()
found = None
for i, candidate in enumerate(words[:SUBSET], 1):
    if bcrypt.checkpw(candidate.encode(), bcrypt_hash):
        found = candidate
        break
dt = time.perf_counter() - start

print(f"Résultat : {found!r} après {i} essais en {dt:.2f}s")
print(f"Débit bcrypt(rounds={ROUNDS}) ≈ {i/dt:.1f} hashs/s")

Cible : $2b$08$/WnlRXslB5u5RqbEsDxGOe7c6pMSoFch06rPcyVsocucUXSOVGceO
Résultat : 'iloveyou' après 105 essais en 2.06s
Débit bcrypt(rounds=8) ≈ 50.9 hashs/s
